# Latent RAG — Factorial Experiment Runner
Compares traditional vs. latent-space RAG across 4 controlled experiments.

| # | Retriever | Generator | Research question |
|---|---|---|---|
| 1 | BGE (dense) | Qwen (text) | Traditional baseline |
| 2 | BGE (dense) | T5Gemma (latent) | Can latent work with good retrieval? |
| 3 | T5Gemma (latent) | Qwen (text) | Does the T5 encoder suck at retrieval? |
| 4 | T5Gemma (latent) | T5Gemma (latent) | Does latent help at all over T5 text? |

## Clone private repo
Use a GitHub Personal Access Token with read access to this repo.

In [3]:
import getpass
import os
import subprocess

os.chdir('/content')

token = getpass.getpass('GitHub token (repo read access): ')
result = subprocess.run(
    ['git', 'clone', f'https://{token}@github.com/zahiTouqan/latent-rag.git'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDOUT:', result.stdout.replace(token, '***'))
    print('STDERR:', result.stderr.replace(token, '***'))
    raise SystemExit('Clone failed')
else:
    print('Clone successful')

STDOUT: ***
STDERR: ***f***a***t***a***l***:*** ***d***e***s***t***i***n***a***t***i***o***n*** ***p***a***t***h*** ***'***l***a***t***e***n***t***-***r***a***g***'*** ***a***l***r***e***a***d***y*** ***e***x***i***s***t***s*** ***a***n***d*** ***i***s*** ***n***o***t*** ***a***n*** ***e***m***p***t***y*** ***d***i***r***e***c***t***o***r***y***.***
***


SystemExit: Clone failed

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
%cd /content/latent-rag
!git fetch
!git checkout v3
!git pull origin v3
!python -m pip install -U pip
!pip install -r requirements.txt
!pip install "datasets>=2.14,<3.0"
!pip install -q --upgrade transformers

In [ ]:
!git pull origin v2

## Download some NQ data
Streams `facebook/dpr` NQs + related documents. Adjust the slice `[:100]` to change dataset size.

In [ ]:
import json, gzip, urllib.request

url = "https://dl.fbaipublicfiles.com/dpr/data/retriever/biencoder-nq-dev.json.gz"
with urllib.request.urlopen(url) as response:
    with gzip.open(response, 'rt', encoding='utf-8') as f:
        data = json.load(f)

sample = data[:100]

Example DPR record:

In [ ]:
import pprint
first = {k: v for k, v in sample[0].items() if k != 'negative_ctxs'}
first['positive_ctxs'] = [{'title': c['title'], 'passage_id': c['passage_id']} for c in first['positive_ctxs']]
pprint.pprint(first, width=120)

## Build mini corpus from NQ passages
Collects all `positive_ctxs` + `hard_negative_ctxs` from the sampled questions.

In [ ]:
import json
from pathlib import Path

CORPUS_OUT = Path("data/passages.jsonl")
CORPUS_OUT.parent.mkdir(parents=True, exist_ok=True)

corpus = {}
for item in sample:
    for p in item["positive_ctxs"] + item["hard_negative_ctxs"]:
        pid = p["passage_id"]
        if pid not in corpus:
            corpus[pid] = {
                "id": pid,
                "text": f"{p['title']}: {p['text']}",
                "doc_id": p["title"]
            }

with CORPUS_OUT.open("w", encoding="utf-8") as f:
    for record in corpus.values():
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(corpus)} passages to {CORPUS_OUT}")

## GPU check & HuggingFace login

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
from huggingface_hub import login
from google.colab import userdata

token = userdata.get('HF_TOKEN')
login(token)

## Build BGE index (for experiments 1 & 2)
Embedding model: `BAAI/bge-base-en-v1.5` — dense retrieval with [CLS]-token pooling.

In [ ]:
BGE_INDEX_DIR = "/content/index_bge"
!python build_index.py --corpus_path {CORPUS_OUT} --index_dir {BGE_INDEX_DIR} --retriever_type bge

In [ ]:
!ls -lh {BGE_INDEX_DIR}

## Build latent (T5) index (for experiments 3 & 4)
Embedding model: `google/t5gemma-2-270m-270m` — mean-pooled encoder latents + safetensors storage.
Stores both FAISS vectors and full sequence latent matrices.
Warning: safetensors file is large (~1.8 GB for 10k passages).

In [ ]:
LATENT_INDEX_DIR = "/content/index_latent"
!python build_index.py --corpus_path {CORPUS_OUT} --index_dir {LATENT_INDEX_DIR} --retriever_type latent

In [ ]:
!ls -lh {LATENT_INDEX_DIR}

## Build eval file
Construct `data/eval.jsonl` from the DPR dev set. `relevant_ids` are document titles for document-level recall.

In [ ]:
import json
from pathlib import Path

EVAL_OUT = Path("data/eval.jsonl")

with EVAL_OUT.open("w", encoding="utf-8") as f:
    for item in sample:
        record = {
            "question": item["question"],
            "answer": item["answers"],
            "relevant_ids": [p["title"] for p in item["positive_ctxs"]]
        }
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(sample)} eval examples to {EVAL_OUT}")

## Run factorial experiments
Each cell runs one experiment. Results are saved to `results/` and can be aggregated after all 4 finish.

### Experiment 1: BGE + Text (traditional baseline)
BGE dense retrieval → prompted text generation with Qwen.

In [ ]:
!python3 evaluate.py \
    --index_dir {BGE_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode bge+text \
    --top_k 5

### Experiment 2: BGE + Latent
BGE dense retrieval → T5Gemma re-encodes retrieved passages on the fly → decoder bypass.

In [ ]:
!python3 evaluate.py \
    --index_dir {BGE_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode bge+latent \
    --top_k 5

### Experiment 3: T5 + Text
T5 encoder mean-pooled retrieval (likely poor) → Qwen text generation.
Isolates retrieval quality of T5 encoder.

In [ ]:
!python3 evaluate.py \
    --index_dir {LATENT_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode t5+text \
    --top_k 5

### Experiment 4: T5 + Latent (full latent pipeline)
T5 encoder retrieval + pre-stored safetensors latents → decoder bypass. The current state of the repo.

In [ ]:
!python3 evaluate.py \
    --index_dir {LATENT_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode t5+latent \
    --top_k 5

## Compare results
Collects all result files from `results/` and displays a side-by-side comparison table.

In [ ]:
import json
from pathlib import Path

results_dir = Path("results")
files = sorted(results_dir.glob("results_*.json"), key=lambda p: p.stat().st_mtime)

COLUMNS = ["em", "f1", "recall@5", "latency_p50_ms", "latency_p95_ms"]

rows = []
for result_file in files:
    with result_file.open() as f:
        data = json.load(f)
    mode = data["config"]["mode"]
    metrics = data["metrics"]
    rows.append((mode, metrics))

if not rows:
    print("No result files found in results/")
else:
    header = f"{'Experiment':<14s}" + "".join(f"{c:>15s}" for c in COLUMNS)
    print(header)
    print("-" * len(header))
    for mode, m in rows:
        vals = "".join(f"{m.get(c, 0):>15.4f}" for c in COLUMNS)
        print(f"{mode:<14s}{vals}")
    if len(rows) < 4:
        print(f"\n({len(rows)}/4 experiments completed)")